Let's configure a simple exporter for the telemetry traces.

In [1]:
import logfire
import os

logfire.configure(
    token=os.environ["LOGFIRE_TOKEN"],
    service_name="ai-assist-demo",
    scrubbing=False,
    send_to_logfire=True,
)

# Getting started

Let's walk through a small but realistic example. The task is **product review sentiment analysis**: given a customer review, predict whether the sentiment is `positive`, `neutral`, or `negative`, and produce a short justification for the call.

We start with a tiny labelled dataset. The `reference` is the ground-truth label.


In [2]:
from lmeh.datatypes import Example

examples = [
    Example(
        inputs={
            "review": "Battery lasts two full days and the screen is gorgeous. Best phone I've owned."
        },
        reference="positive",
    ),
    Example(
        inputs={
            "review": "It works. Setup was fine, nothing surprising, nothing to complain about."
        },
        reference="positive",
    ),
    Example(
        inputs={"review": "Stopped charging after three weeks. Support never replied. Avoid."},
        reference="negative",
    ),
    Example(
        inputs={
            "review": "    Camera   is   AMAZING!!!   colors pop, low-light is great.\n\n\nHighly recommend."
        },
        reference="positive",
    ),
    Example(
        inputs={
            "review": "Arrived on time. Packaging was a bit beaten up but the product itself looks ok."
        },
        reference="neutral",
    ),
]

Now we define the target function — the **thing we actually want to evaluate**. Note that it is not just a thin wrapper around the call to the language model `complete()`: there can be real pre- and post-processing around the model call. That's intentional. In production, what users hit is rarely the raw completion; it's the **completion plus the scaffold around it**. The harness lets us evaluate that full product.

The function must adhere to the `TargetFunction` protocol: take its **named inputs**, the **prompt template**, and an **LM config**, then return whatever the downstream scorers should consume. The harness unpacks `Example.inputs` as keyword arguments, so the parameter names must match the dict keys.
    

In [3]:
import re
from typing import Literal

from lmdk import complete, render_template
from pydantic import BaseModel, Field

from lmeh.datatypes import LMConfig

ALLOWED_LABELS = {"positive", "neutral", "negative"}
MAX_REVIEW_CHARS = 500

def classify_sentiment(review: str, prompt_template: str, config: LMConfig) -> dict:
    # Pre-processing: normalise whitespace and cap length
    cleaned = re.sub(r"\s+", " ", review).strip()
    if len(cleaned) > MAX_REVIEW_CHARS:
        cleaned = cleaned[:MAX_REVIEW_CHARS] + "…"

    # Define the output schema expected from the model
    class Output(BaseModel):
        sentiment: Literal["positive", "neutral", "negative"]
        reason: str = Field(description="One short sentence justifying the sentiment label.")

    # Call the model with lmdk.complete
    prompt = render_template(template=prompt_template, REVIEW=cleaned)
    response = complete(
        model=config.model,
        generation_kwargs=config.generation_kwargs,
        prompt=prompt,
        output_schema=Output,
    )

    # Post-processing: normalize the label and fall back to"neutral"
    label = (response.output.sentiment or "").strip().lower()
    if label not in ALLOWED_LABELS:
        label = "neutral"

    # Return arbitrary format that will be downstream consumed
    return {"sentiment": label, "reason": response.output.reason.strip()}

Now, lets define the moving parts under test. The two sweepable axes are the **prompt template** and the **LM config** (model, generation kwargs).


In [4]:
prompt_template = """Your task is to classify the sentiment of this product review:

{{ REVIEW }}

The possible labels are 'positive', 'neutral' and 'negative'
"""

config = LMConfig(
    model="mistral:mistral-small-latest",
    generation_kwargs={"temperature": 0.2},
)

With these ingredients, we can already run a trial: execute the target function on one example with the config under test.


In [5]:
from lmeh.execution import run_trial

trial = run_trial(
    target=classify_sentiment,
    prompt_template=prompt_template,
    config=config,
    example=examples[0],
)

trial.output

Logfire project URL: https://logfire-eu.pydantic.dev/nachollorca/ai-assist-demo

16:59:10.449 chat mistral-small-latest


{'sentiment': 'positive',
 'reason': "The review expresses strong satisfaction with the phone's battery life, screen quality, and overall performance."}

Now that our trial ran successfully, we can jump into quality measurements.

First, a simple deterministic check: does the predicted label match the ground truth? No LLM judge needed — a `ProgrammaticMetric` whose scorer follows the `ProgrammaticScorer` protocol is the right tool. We declare the value space with an `Ordinal` scale.

In [6]:
from lmeh.datatypes import Ordinal, ProgrammaticMetric, RawScore

def label_match(output: dict, example: Example) -> RawScore:
    predicted = output["sentiment"]
    expected = example.reference
    if predicted == expected:
        return RawScore(raw=True, reason=f"Predicted '{predicted}' matches reference.")
    return RawScore(
        raw=False,
        reason=f"Predicted '{predicted}', expected '{expected}'.",
    )

label_correctness = ProgrammaticMetric(
    name="label_correctness",
    description="Whether the predicted sentiment label matches the ground-truth label.",
    scale=Ordinal(levels=[False, True]),
    scorer=label_match,
)

Now we score the trial against the metric.

In [7]:
from lmeh.execution import score_metric

scoring = score_metric(trial=trial, metric=label_correctness)
scoring.score

Score(raw=True, normalized=1.0, reason="Predicted 'positive' matches reference.")

The label check is cheap and exact, but it tells us nothing about the *reason* the model produced — and a good sentiment classifier should be able to justify its call. That's a subjective, open-ended judgement with no ground truth, which is exactly where an LLM judge earns its keep.

We define an `LLMJudgeMetric` that grades the quality of the justification. It carries its own `LMConfig` and judge prompt template. We use the built-in `default_llm_judge`, which feeds the judge the rendered prompt, the target's output, the reference, and the metric description.


In [8]:
from lmeh.datatypes import LLMJudgeMetric
from lmeh.judges import default_llm_judge

description = """Rate how well the 'reason' field justifies the predicted sentiment label given the original review. A good reason cites concrete cues from the review and is consistent with the predicted label. Score 'good' if the justification is grounded and coherent, 'poor' otherwise."""

reason_quality = LLMJudgeMetric(
    name="reason_quality",
    description=description,
    scale=Ordinal(levels=["poor", "good"]),
    scorer=default_llm_judge,
    config=LMConfig(
        model="mistral:mistral-medium-latest",
        generation_kwargs={"temperature": 0.1},
    ),
)

Let's see what the judge thinks about the system output on our first trial.


In [9]:
judge_scoring = score_metric(trial=trial, metric=reason_quality)
judge_scoring.score

16:59:11.035 chat mistral-medium-latest


Score(raw='good', normalized=1.0, reason="The 'reason' field explicitly cites concrete cues from the review ('battery life', 'screen quality', and 'overall performance') and aligns them with the predicted 'positive' sentiment label, demonstrating a grounded and coherent justification.")

Finally, we can wire it all together into an experiment: run `classify_sentiment` against every example and score both metrics on each output.


In [10]:
from lmeh.datatypes import Experiment
from lmeh.execution import run_experiment

experiment = Experiment(
    name="sentiment-baseline",
    target=classify_sentiment,
    prompt_template=prompt_template,
    config=config,
)

results = run_experiment(
    experiment=experiment,
    examples=examples,
    metrics=[label_correctness, reason_quality],
    workers=5,
)

16:59:12.462 chat mistral-small-latest
16:59:12.465 chat mistral-small-latest
16:59:12.467 chat mistral-small-latest
16:59:12.470 chat mistral-small-latest
16:59:12.478 chat mistral-small-latest
16:59:12.972 chat mistral-medium-latest
16:59:12.980 chat mistral-medium-latest
16:59:12.982 chat mistral-medium-latest
16:59:12.984 chat mistral-medium-latest
16:59:13.012 chat mistral-medium-latest


And a rendering utility renders the results.

In [11]:
from lmeh.rendering import render_run
from IPython.display import Markdown, display

report = render_run(results)
display(Markdown(report))
display(Markdown("---"))

### Quality

Headline score is the mean across per-metric means (each metric weighted equally).

- **Overall**: 0.80 (2 metrics)

Each cell is `mean ± sd (n)`. `Score` aggregates per-example means (dispersion = dataset heterogeneity); `Replicate noise` is the average within-cell SD across replicates (dispersion = measurement instability).

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 0.80 ± 0.45 (n=5) | 0.00 ± 0.00 (n=5) |
| `reason_quality` | 0.80 ± 0.45 (n=5) | 0.00 ± 0.00 (n=5) |

### Reliability

Trial failures count against the run (the target is under evaluation); scorer failures are excluded from quality aggregates.

- **Trials**: 5 successful / 5 total
- **Trial failure rate**: 0.00%
- **Scorings**: 10 successful / 10 total
- **Scoring failure rate**: 0.00%

### Weak examples

1. **mean 0.00** — `{'review': 'It works. Setup was fine, nothing surprising, nothing to complain about.'}`
   - `label_correctness`: 0.00 — Predicted 'neutral', expected 'positive'.
   - `reason_quality`: 0.00 — The reason provided ('The review expresses satisfaction with no complaints, indicating a neutral to positive sentiment.') is inconsistent with the predicted label 'neutral'. The review contains positive cues like 'It works' and 'nothing to complain about', which align more with 'positive' than 'neutral'. The justification does not clearly support the chosen label.

### Telemetry

Totals across successful trials only.

- **Total latency**: 2.567 s
- **Total output tokens**: 148
- **Throughput**: 57.6 tok/s

---

## Raising the bar before we optimize

The baseline already scores well — the five reviews above are easy, and the vague prompt (*"classify the sentiment, the labels are positive / neutral / negative"*) captures everything they need. There's nothing for the optimizer to win.

To create real headroom, we extend the dataset with reviews whose **correct label depends on a labelling rubric the baseline prompt never states**. The optimizer can only rewrite the prompt template, and it sees the weakest examples plus the judge's reasoning each step — so it can *infer* the rubric from the failures and encode it into a better prompt. That's the loop we want to show off.

The rubric, applied consistently across the new examples:

1. **Aspect scoping** — judge the sentiment toward the **product itself**. Slow shipping, a crushed box, or great/poor customer service do **not** change the label. (A vague prompt averages product and logistics together and mislabels.)
2. **Sarcasm & negation** — judge the *intent*, not the surface words. `"Worth every penny, truly"` about a watch that died is **negative**; `"not the premium knife they promise"` is **negative** despite the positive vocabulary.
3. **`neutral` means genuinely mixed or purely factual** — real pros and cons that roughly balance, or a description with no evaluation. A *minor* gripe inside an otherwise glowing review stays **positive** (the counter-case below guards against over-applying `neutral`).

**What success looks like:** the baseline prompt should now score noticeably below 1.0 on these (conflating logistics with the product, getting fooled by sarcasm, scattering the mixed cases). A well-optimized prompt — one that spells out "focus on the product, watch for sarcasm/negation, reserve neutral for balanced or factual reviews" — should recover most of that gap. The `best` score climbing meaningfully above the `baseline` score is the demo paying off.

In [12]:
ambiguous_examples = [
    # Rule 1 — aspect scoping: the PRODUCT is great, only shipping/packaging is bad.
    Example(
        inputs={
            "review": "Took almost a month to arrive and the box was crushed in transit, but the espresso machine itself is superb — rich crema every single morning."
        },
        reference="positive",
    ),
    # Rule 1 — aspect scoping: shipping/service is great, the PRODUCT is the letdown.
    Example(
        inputs={
            "review": "Lightning-fast shipping and the courier was lovely, but the earbuds crackle in one ear and the battery dies within an hour. The product is a letdown."
        },
        reference="negative",
    ),
    # Rule 2 — sarcasm: every surface word is positive, the intent is the opposite.
    Example(
        inputs={
            "review": "Oh brilliant, another 'waterproof' watch that died the first time it saw rain. Worth every penny, truly."
        },
        reference="negative",
    ),
    # Rule 2 — negation: positive-sounding vocabulary, flipped by 'not'.
    Example(
        inputs={
            "review": "Don't believe the glowing reviews — this is not the durable, premium knife they promise. It chipped on day two."
        },
        reference="negative",
    ),
    # Rule 3 — neutral = genuinely mixed, real pros and cons that roughly balance.
    Example(
        inputs={
            "review": "The fabric is soft and the colour is lovely, but it shrank in the first wash and the stitching is already loose. Hard to call it good or bad."
        },
        reference="neutral",
    ),
    # Rule 3 — neutral = purely factual, no evaluation at all.
    Example(
        inputs={"review": "It's a 2-metre USB-C cable, black, exactly as pictured. Bought it to replace a lost one."},
        reference="neutral",
    ),
    # Counter-case: a minor gripe must NOT tip a clearly positive review into 'neutral'.
    Example(
        inputs={
            "review": "Runs a touch warm under heavy load, but honestly this laptop is phenomenal — fast, silent, and the display is stunning. No regrets."
        },
        reference="positive",
    ),
]

examples = examples + ambiguous_examples

The harness can also **search for a better prompt template**. `optimize()` evaluates the experiment's current template as a baseline, then repeatedly asks a *proposer* model to rewrite the template from the full trajectory and re-scores each candidate. The archive keeps every checkpoint; `OptimizationResult.best` is the highest-scoring step (not necessarily the last).

In [13]:
from lmeh.optimization import optimize

optimization = optimize(
    experiment=experiment,
    examples=examples,
    metrics=[label_correctness, reason_quality],
    proposer_config=LMConfig(
        model="vertex:gemini-3.5-flash",
        generation_kwargs={"temperature": 0.3},
    ),
    steps=4,
    workers=5,
)

16:59:15.293 chat mistral-small-latest
16:59:15.295 chat mistral-small-latest
16:59:15.297 chat mistral-small-latest
16:59:15.299 chat mistral-small-latest
16:59:15.301 chat mistral-small-latest
16:59:15.738 chat mistral-small-latest
16:59:15.794 chat mistral-small-latest
16:59:15.797 chat mistral-small-latest
16:59:15.802 chat mistral-small-latest
16:59:15.852 chat mistral-small-latest
16:59:16.278 chat mistral-small-latest
16:59:16.286 chat mistral-small-latest
16:59:16.306 chat mistral-medium-latest
16:59:16.319 chat mistral-medium-latest
16:59:16.354 chat mistral-medium-latest
16:59:16.763 chat mistral-medium-latest
16:59:16.865 chat mistral-medium-latest
16:59:17.402 chat mistral-medium-latest
16:59:17.964 chat mistral-medium-latest
16:59:18.472 chat mistral-medium-latest
16:59:18.558 chat mistral-medium-latest
16:59:18.567 chat mistral-medium-latest
16:59:19.561 chat mistral-medium-latest
16:59:19.631 chat mistral-medium-latest
16:59:27.124 chat gemini-3.5-flash
16:59:28.719 chat

In [15]:
baseline = optimization.steps[0]
best = optimization.best

print(f"Baseline score: {baseline.score:.4f}")
print(f"Best score:     {best.score:.4f} (step {optimization.steps.index(best)})")
print()
print("Best prompt template:")
print(best.prompt_template)

Baseline score: 0.9583
Best score:     0.9583 (step 0)

Best prompt template:
Your task is to classify the sentiment of this product review:

{{ REVIEW }}

The possible labels are 'positive', 'neutral' and 'negative'

